In [17]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster, heat_map
from folium.plugins import HeatMap
from folium.plugins import HeatMapWithTime

# **Data Loading:**

- Vancouver Crime Data
- Austin Crime Data
- Vancouver By-law (311) Data

In [7]:
# Paths
vancouver_path = '/Users/p.silva/Documents/GitHub/EC-Project/data/raw/vancouver_crime_data.csv'
austin_texas = '/Users/p.silva/Documents/GitHub/EC-Project/data/raw/austin_texas_crime_data.csv'
service_requests_path = '/Users/p.silva/Documents/GitHub/EC-Project/data/raw/3-1-1-service-requests.csv'
austin_texas_full = '/Users/p.silva/Documents/GitHub/EC-Project/data/raw/austin_full.csv'

# Dataframes:
# vancouver and austin data crime
# 3-1-1 service requests
df_vancouver = pd.read_csv(vancouver_path)
df_austin = pd.read_csv(austin_texas)
df_service_requests = pd.read_csv(service_requests_path, delimiter=';')
df_austin_full = pd.read_csv(austin_texas_full, low_memory=False)

In [4]:
df_austin.columns

Index(['Incident Number', 'Highest Offense Description',
       'Highest Offense Code', 'Family Violence', 'Occurred Date Time',
       'Occurred Date', 'Occurred Time', 'Report Date Time', 'Report Date',
       'Report Time', 'Location Type', 'Council District', 'APD Sector',
       'APD District', 'Clearance Status', 'Clearance Date', 'UCR Category',
       'Category Description', 'Census Block Group'],
      dtype='str')

In [3]:
df_vancouver.columns

Index(['TYPE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE', 'HUNDRED_BLOCK',
       'NEIGHBOURHOOD', 'X', 'Y'],
      dtype='str')

In [6]:
df_service_requests.columns

Index(['Department', 'Service request type', 'Status', 'Closure reason',
       'Service request open timestamp', 'Service request close date',
       'Last modified timestamp', 'Address', 'Local area', 'Channel',
       'Latitude', 'Longitude', 'geom'],
      dtype='str')

In [7]:
df_austin_full.columns

Index(['Incident Number', 'Highest Offense Description',
       'Highest Offense Code', 'Family Violence', 'Occurred Date Time',
       'Occurred Date', 'Occurred Time', 'Report Date Time', 'Report Date',
       'Report Time', 'Location Type', 'Address', 'Zip Code',
       'Council District', 'APD Sector', 'APD District', 'PRA', 'Census Tract',
       'Clearance Status', 'Clearance Date', 'UCR Category',
       'Category Description', 'X-coordinate', 'Y-coordinate', 'Latitude',
       'Longitude', 'Location'],
      dtype='str')

# **Playing with Maps without Geojson Data:**

### **Simple Interactive Map:**

In [13]:
df = df_vancouver
df_month = df[(df['YEAR'] >= 2023) & (df['YEAR'] <= 2026)].copy()
df_clean = df_month[
    (df_month['X'].notnull()) &
    (df_month['Y'].notnull()) &
    (df_month['X'] != 0.0) &
    (df_month['Y'] != 0.0)
].copy()

gdf = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean['X'], df_clean['Y']),
    crs="EPSG:26910" # UTM Zone 10N
)

gdf = gdf.to_crs("EPSG:4326")
map = folium.Map(location=[49.2827, -123.1207], zoom_start=12, tiles='CartoDB positron')

cluster = MarkerCluster().add_to(map)

for idx, row in gdf.iterrows():
    # Extract X,Y coordinates
    lat = row.geometry.y
    lon = row.geometry.x

    # Red point by each crime
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color="crimson",
        fill=True,
        fill_opacity=0.7,
        popup=f"<b>Type:</b> {row['TYPE']}<br><b>Block:</b> {row['HUNDRED_BLOCK']}"
    ).add_to(cluster)

In [ ]:
map

### **Heatmap - Kernel Density Estimation**

In [18]:
heat_map = folium.Map(location=[49.2827, -123.1207], zoom_start=12, tiles='CartoDB dark_matter')

coordinates = [[row.geometry.y, row.geometry.x] for index, row in gdf.iterrows()]
HeatMap(
    coordinates,
    radius=15,
    blur=20,
    gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 0.8: 'orange', 1.0: 'red'}
).add_to(heat_map)

In [ ]:
heat_map

### **Animated Temporal Map:**

In [23]:
df['YEAR'] = pd.to_numeric(df['YEAR'], errors='coerce')
df['MONTH'] = pd.to_numeric(df['MONTH'], errors='coerce')
df['DAY'] = pd.to_numeric(df['DAY'], errors='coerce')

df_january = df[(df['YEAR'] >= 2023) & (df['YEAR'] <= 2026) & (df['MONTH'] == 1)].copy()

# Prevents the map from breaking or centering on [0,0] (Equator)
df_january = df_january[
    (df_january['X'].notnull()) &
    (df_january['Y'].notnull()) &
    (df_january['X'] != 0.0) &
    (df_january['Y'] != 0.0)
].copy()

# Initialize the base map
time_map = folium.Map(location=[49.2827, -123.1207], zoom_start=12, tiles='CartoDB dark_matter')


# Sort days for chronological order
days_in_month = sorted(df_january['DAY'].dropna().unique())

data_by_time = []
time_index = []

for day in days_in_month:
    # Filter records for the specific day
    df_day = df_january[df_january['DAY'] == day]

    # Convert X/Y (UTM Zone 10N) to Lat/Lon (WGS84)
    gdf_day = gpd.GeoDataFrame(
        df_day,
        geometry=gpd.points_from_xy(df_day['X'], df_day['Y']),
        crs="EPSG:26910"
    ).to_crs("EPSG:4326")

    # Extract coordinates
    coords_day = [[row.geometry.y, row.geometry.x] for index, row in gdf_day.iterrows()]

    data_by_time.append(coords_day)
    time_index.append(f"January {int(day)}")

# Add the animation layer to the map
HeatMapWithTime(
    data=data_by_time,
    index=time_index,
    auto_play=True,
    radius=20,
    display_index=True
).add_to(time_map)

In [ ]:
time_map